# Function & Tool Calling — Teaching Notebook

**Case study:** mFinAgent — an NSE stock-analytics agent

This notebook implements the concepts from the tutorial using the **LangChain DeepAgents** framework with **Groq** as the LLM provider. By the end you'll have a working agent that:

- Resolves a fuzzy company name to an NSE ticker (with typo tolerance)
- Fetches real monthly returns from yfinance
- Computes basic statistics (mean, volatility, probability of monthly return > threshold)
- Runs the full tool-calling loop **inside the framework** — the loop is hidden, but the tools you write are 100% yours


## Prerequisites

Install the required packages:

```bash
pip install deepagents langchain langchain-groq yfinance pandas
```

Set your Groq API key as an environment variable:

```bash
export GROQ_API_KEY="gsk_..."
```

Get a free Groq API key at https://console.groq.com.

Then run the cells below in order.

In [1]:
!pip install deepagents langchain langchain-groq yfinance pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.5/221.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 13.7 MB/s eta 0:00:00


In [11]:
# ── Imports ──
import os
import math
import json
import difflib
from getpass import getpass
import numpy as np

import pandas as pd
import yfinance as yf

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from deepagents import create_deep_agent

In [3]:
os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY:")

GROQ_API_KEY:··········


---

## Section 1: Why Tool Calling?

A standalone LLM is a closed system — it can only output text, and only from what it learned during training. If you ask it for HDFC Bank's last 6-month return, it has two options:

1. **Refuse**: *"I don't have access to current market data."* — modern models prefer this.
2. **Hallucinate**: confidently invent numbers that *sound* right but aren't real.

Either way, the user gets nothing useful.

**Tool calling** lets the LLM reach outside its weights — into databases, APIs, and live data feeds. Crucially, the LLM doesn't execute anything itself; it emits a *structured request* to call a function, and **your code** runs the function and returns the result. Let's see this concretely.

In [5]:
# ── Demo: LLM without tools ──
# Ask a current-data question to a model that has no tool access.
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
resp = llm.invoke("What was return from HDFCBANK in last 6 months?")
print(resp.content)

I don't have access to real-time data or specific information about HDFC Bank's current performance. However, I can suggest some ways for you to find the information you're looking for:

1. Check the bank's official website: You can visit HDFC Bank's official website and look for the "Investor Relations" or "Share Price" section, which may provide information on the bank's recent performance.
2. Use a financial website or platform: Websites like Bloomberg, Yahoo Finance, or Google Finance provide real-time data on stock prices and performance. You can search for HDFC Bank's stock symbol (HDFCBANK.NS) to get the latest information.
3. Consult a financial advisor: If you're looking for personalized investment advice or more detailed information about HDFC Bank's performance, you may want to consult a financial advisor or a broker.

Please note that past performance is not a guarantee of future results, and it's always important to do your own research and consider multiple sources before

---

## Section 2: The Tool-Calling Loop

Every tool-using interaction is the same four-step loop, with a decision in the middle:

```
[User msg + tools] → [LLM decides] → ◇ Tool call?
                                       │
                            ┌──────────┴──────────┐
                          YES                    NO (text reply)
                            ↓                     ↓
                    [Execute tool]     [Final reply, exit loop]
                            ↓
                    [Feed result back to LLM] ↺ loop
```

Three properties to remember:

1. **The LLM never executes anything itself.** It can only request a call.
2. **The loop is driven by the runtime.** Each round is a separate API call.
3. **The conversation grows monotonically.** Both the assistant's request and your reply are appended.

With **DeepAgents**, this loop runs *inside* `create_deep_agent` — you write the tools, the framework handles the rest.

### Tool 1: `fetch_monthly_returns`

Pulls actual monthly close-to-close returns from yfinance. This is where the LLM gets *grounded* in real data — every numerical answer it gives traces back to a row in this output.

Returns a dict with:
- `ticker` — the symbol queried
- `returns_pct` — list of `{month, return_pct}` entries
- `total_return_pct` — cumulative return over the window

In [27]:
@tool
def get_stock_returns(ticker: str, month:str ):

    '''
        ticker: the stock ticker name
        month: number of month for which the stock trading data to used e.g. 1 or 2 or 3 etc.
        input string ticker is The correct ticker symbol for the stock in Nation Stock Exchange (NSE) India.
    '''

    # Get ticker name correctly
    msft = yf.Ticker(ticker.split(".")[0] + ".NS")

    # get historical market data
    hist = msft.history(period = month + "mo" )

    # Compute the market data
    hist['daily_changes']  = (hist['Close'] - hist['Open']) * 100 / hist['Open']

    # Compute different statistics
    total_gain = (hist.iloc[-1]["Close"] - hist.iloc[0]["Open"] ) * 100 / hist.iloc[0]["Open"]
    avg_daily_changes = np.mean(hist['daily_changes'])
    std_daily_changes = np.std(hist['daily_changes'])

    stock_stats = {'total_gain_in_percentage': round(total_gain, 3),
                   'average_daily_changes_in_percentage': round(avg_daily_changes, 3),
                   'std_daily_changes_in_percentage': round(std_daily_changes, 3)}

    return json.dumps(stock_stats)

In [28]:
# Try it on HDFC Bank
result = get_stock_returns.invoke({"ticker": "HDFCBANK.NS", "month": "6"})
print(json.dumps(result, indent=2, default=str))

"{\"total_gain_in_percentage\": -24.693, \"average_daily_changes_in_percentage\": 0.046, \"std_daily_changes_in_percentage\": 1.229}"


---

## Section 3: Build the Agent

With the three tools defined, building the agent is a single call. The system prompt nudges the model to use the tools in the right order: **lookup → fetch → compute**.

In [29]:
SYSTEM = """You are mFinAgent, an Indian stock-analytics assistant.

When the user asks about a company:
1. Call `lookup_ticker` first to resolve the name to an NSE symbol.
   If multiple candidates score similarly, ask the user to disambiguate.
2. Call `fetch_monthly_returns` with the resolved ticker and time window.
3. If the user asks about risk, volatility, or probabilities, call
   `compute_statistics` on the returns list.
4. Always cite the actual numbers returned by the tools — never make them up.
"""

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

agent = create_deep_agent(
    model=model,
    tools=[get_stock_returns],
    system_prompt=SYSTEM,
)

print("Agent ready.")

Agent ready.


### Run a query

Invoke the agent with a natural-language question. Internally it will run the four-step loop until the model produces a final answer.

In [30]:
def ask(query):
  result = agent.invoke({
      "messages": [{
          "role": "user",
          "content": query
      }]
  })

  print("=== FINAL ANSWER ===")
  print(result["messages"][-1].content)

In [32]:
ask("What was return from HDFCBANK in last 6 months?")

=== FINAL ANSWER ===
HDFCBANK’s total return over the past 6 months was **‑24.69 %**. (Average daily change ≈ 0.046 % with a daily volatility of ≈ 1.23 %.)


In [33]:
ask("between HDFCBANK and YESBANK, which stock has given better returns in last 3 months?")

=== FINAL ANSWER ===
YESBANK outperformed HDFCBANK over the past 3 months.

- **HDFCBANK:** ‑14.51 % total return  
- **YESBANK:** +16.27 % total return  

Thus, YESBANK delivered the better return in the last three months.
